In [ ]:
#ETL conectado ao Trello para extrair dados de cards e calcular lead time, porém não há dados nas etapas

import requests
import pandas as pd
from datetime import datetime

# 1. Credenciais de acesso à API do Trello
NOME_QUADRO = "CTI "


# Funcao para fazer requisições GET à API do Trello
def trello_get(endpoint, params=None):
    base_url = "https://api.trello.com/1"
    full_url = f"{base_url}/{endpoint}"
    auth = {'key': API_KEY, 'token': API_TOKEN}
    if params:
        auth.update(params)
    response = requests.get(full_url, params=auth)
    response.raise_for_status()
    return response.json()

# Buscar o quadro pelo nome e obter seus cards
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)
if not board:
    raise Exception(f"Quadro '{NOME_QUADRO}' não encontrado")

print(f"✅ Conectado: {board['name']} (ID: {board['id']})")

cards = trello_get(f"boards/{board['id']}/cards")
print(f"✅ {len(cards)} cards encontrados")

# ETL - Extrair ações de cada card (movimentações entre listas)
acoes = []
for card in cards:
    card_actions = trello_get(f"cards/{card['id']}/actions", params={'filter': 'updateCard'})
    for act in card_actions:
        acoes.append({
            'card_id': card['id'],
            'card_name': card['name'],
            'action_date': act['date'],
            'list_before': act['data'].get('listBefore', {}).get('name'),
            'list_after': act['data'].get('listAfter', {}).get('name'),
            'action_type': act['type']
        })

df_acoes = pd.DataFrame(acoes)
print(f"✅ {len(df_acoes)} ações registradas")

# Calcular lead time (dias entre criação e conclusão)
lead_times = []
for card in cards:
    # Aproximação: usa 'dateLastActivity' como criação (não é 100% preciso)
    # Ideal: buscar a PRIMEIRA ação do card (tipo 'createCard')
    data_criacao = pd.to_datetime(card['dateLastActivity'])
    
    # Busca quando o card entrou na lista "Concluido"
    mov_concluido = df_acoes[
        (df_acoes['card_id'] == card['id']) & 
        (df_acoes['list_after'] == 'Concluido')
    ]
    
    if not mov_concluido.empty:
        data_conclusao = pd.to_datetime(mov_concluido.iloc[0]['action_date'])
        lead_time_dias = (data_conclusao - data_criacao).days
        status = 'Concluído'
    else:
        data_conclusao = None
        lead_time_dias = None
        status = 'Não concluído'
    
    lead_times.append({
        'card_id': card['id'],
        'card_name': card['name'],
        'data_criacao': data_criacao,
        'data_conclusao': data_conclusao,
        'lead_time_dias': lead_time_dias,
        'status': status
    })

df_lead = pd.DataFrame(lead_times)

# Exportar dados para CSV - dados brutos csv
# Cards (dados brutos)
df_cards = pd.DataFrame(cards)
df_cards.to_csv('cards_trello.csv', index=False)

# Ações (movimentações)
df_acoes.to_csv('acoes_trello.csv', index=False)

# Lead time (com card_id para relação)
df_lead.to_csv('lead_time.csv', index=False)

print("\n✅ Arquivos gerados:")
print("  - cards_trello.csv")
print("  - acoes_trello.csv")
print("  - lead_time.csv")

print("\n📊 Amostra do lead_time:")
print(df_lead[['card_name', 'lead_time_dias', 'status']].head(10))

✅ Conectado: CTI  (ID: 69fd20465fb2b7d1fb85e050)
✅ 0 cards encontrados
✅ 0 ações registradas

✅ Arquivos gerados:
  - cards_trello.csv
  - acoes_trello.csv
  - lead_time.csv

📊 Amostra do lead_time:


KeyError: "None of [Index(['card_name', 'lead_time_dias', 'status'], dtype='str')] are in the [columns]"

In [ ]:
import requests
import pandas as pd
from datetime import datetime

# 1. Credenciais de acesso à API do Trello



# Funcao para fazer requisições GET à API do Trello
def trello_get(endpoint, params=None):
    base_url = "https://api.trello.com/1"
    full_url = f"{base_url}/{endpoint}"
    auth = {'key': API_KEY, 'token': API_TOKEN}
    if params:
        auth.update(params)
    response = requests.get(full_url, params=auth)
    response.raise_for_status()
    return response.json()

# Nova Funcao: Busca a data de criação real do card
def get_card_created_date(card_id):
    """Retorna a data de criação do card a partir da ação 'createCard'.
       Se não encontrar, retorna None."""
    try:
        actions = trello_get(f"cards/{card_id}/actions", params={'filter': 'createCard'})
        if actions:
            return pd.to_datetime(actions[0]['date'])
        else:
            return None
    except Exception as e:
        print(f"⚠️ Erro ao buscar criação do card {card_id}: {e}")
        return None

# Buscar o quadro pelo nome e obter seus cards
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)
if not board:
    raise Exception(f"Quadro '{NOME_QUADRO}' não encontrado")

print(f"✅ Conectado: {board['name']} (ID: {board['id']})")

cards = trello_get(f"boards/{board['id']}/cards")
print(f"✅ {len(cards)} cards encontrados")

# Extrair ações de cada card (movimentações entre listas)

acoes = []
for card in cards:
    card_actions = trello_get(f"cards/{card['id']}/actions", params={'filter': 'updateCard'})
    for act in card_actions:
        acoes.append({
            'card_id': card['id'],
            'card_name': card['name'],
            'action_date': act['date'],
            'list_before': act['data'].get('listBefore', {}).get('name'),
            'list_after': act['data'].get('listAfter', {}).get('name'),
            'action_type': act['type']
        })

df_acoes = pd.DataFrame(acoes)
print(f"✅ {len(df_acoes)} ações registradas")

# Calcular lead time (com data de criação real)

lead_times = []
cards_sem_data_criacao = 0

for card in cards:
    # 1. Tenta obter a data de criação REAL via ação 'createCard'
    data_criacao = get_card_created_date(card['id'])
    
    # 2. Se não encontrou, usa 'dateLastActivity' como fallback (e avisa)
    if data_criacao is None:
        data_criacao = pd.to_datetime(card['dateLastActivity'])
        cards_sem_data_criacao += 1
        print(f"⚠️ Card '{card['name']}' sem ação createCard. Usando dateLastActivity como aproximação.")
    
    # 3. Busca quando o card entrou na lista "Concluido"
    mov_concluido = df_acoes[
        (df_acoes['card_id'] == card['id']) & 
        (df_acoes['list_after'] == 'Concluido')
    ]
    
    if not mov_concluido.empty:
        data_conclusao = pd.to_datetime(mov_concluido.iloc[0]['action_date'])
        lead_time_dias = (data_conclusao - data_criacao).days
        status = 'Concluído'
    else:
        data_conclusao = None
        lead_time_dias = None
        status = 'Não concluído'
    
    lead_times.append({
        'card_id': card['id'],
        'card_name': card['name'],
        'data_criacao': data_criacao,
        'data_conclusao': data_conclusao,
        'lead_time_dias': lead_time_dias,
        'status': status
    })

df_lead = pd.DataFrame(lead_times)

# Relatório de qualidade dos dados
print(f"\n📊 Qualidade dos dados:")
print(f"  - Cards com data de criação real (createCard): {len(cards) - cards_sem_data_criacao}")
print(f"  - Cards com data aproximada (dateLastActivity): {cards_sem_data_criacao}")

# EXPORTAR ARQUIVOS
df_cards = pd.DataFrame(cards)
df_cards.to_csv('cards_trello.csv', index=False)

df_acoes.to_csv('acoes_trello.csv', index=False)

df_lead.to_csv('lead_time.csv', index=False)

print("\n✅ Arquivos gerados:")
print("  - cards_trello.csv")
print("  - acoes_trello.csv")
print("  - lead_time.csv")

print("\n📊 Amostra do lead_time:")
print(df_lead[['card_name', 'lead_time_dias', 'status']].head(10))

In [ ]:
# verificar se a conexão com a API do Trello está funcionando e listar os quadros do usuário

import requests



url = "https://api.trello.com/1/members/me/boards"
params = {'key': API_KEY, 'token': API_TOKEN}

response = requests.get(url, params=params)
boards = response.json()

print("📋 Seus quadros no Trello:")
for board in boards:
    print(f"  - Nome: '{board['name']}' (ID: {board['id']})")

📋 Seus quadros no Trello:
  - Nome: 'CTI ' (ID: 69fd20465fb2b7d1fb85e050)
  - Nome: 'Desafio Integrador 28' (ID: 69b88b24649945fe7312a2bc)
  - Nome: 'Projeto CTI' (ID: 69fd1a648ced6e351dc9282d)
  - Nome: 'Senai Notes - G5.for you' (ID: 681e971bd9a39734b445f278)
